# DevReady AI — 전체 서버 (Colab + Google Drive)

`server.py`를 통째로 띄워 **모든 엔드포인트**를 제공한다. 코드·RAG 인덱스·`train.jsonl`은 공유 **Drive 폴더**에서, 베이스 모델과 **LoRA 어댑터는 HuggingFace**에서 받는다.

### 사전 준비 (한 번만)
1. 공유받은 **`AI-캡스톤(비상업용)`** 폴더(또는 그 폴더가 들어있는 팀 폴더)를 **내 드라이브에 바로가기 추가** — cell 1이 위치를 알아서 찾으니 경로 수정은 필요 없다.
2. **🔑 Secrets** 패널에 **`HF_TOKEN`** 등록(어댑터 레포 Read 토큰) + Notebook access ON

### 실행
3. **런타임 → GPU(T4)** 선택 → **모두 실행(Run all)**

> 첫 실행은 베이스 모델(~15GB) 다운로드로 **7~9분** 걸린다(정상). 마지막 셀이 `https://....trycloudflare.com` URL을 출력. Colab 세션제.

## 1. Google Drive 마운트 + 폴더 자동탐색
바로가기를 내 드라이브 바로 아래에 뒀든, 팀 폴더 한 단계 안에 뒀든 자동으로 찾는다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
TARGET = "AI-캡스톤(비상업용)"
ROOT = "/content/drive/MyDrive"

# 내 드라이브 최상위 + 한 단계 하위 폴더까지 자동 탐색
candidates = [os.path.join(ROOT, TARGET)]
try:
    for d in os.listdir(ROOT):
        p = os.path.join(ROOT, d)
        if os.path.isdir(p):
            candidates.append(os.path.join(p, TARGET))
except Exception:
    pass

DRIVE_DIR = next((c for c in candidates if os.path.isdir(c)), None)
assert DRIVE_DIR, (f"'{TARGET}' 폴더를 못 찾음 — 공유 폴더를 '내 드라이브에 바로가기 추가' 했는지 확인.\n"
                   f"내 드라이브 목록: {os.listdir(ROOT)}")
print(">>> 찾음:", DRIVE_DIR)
print(">>> 내용:", sorted(os.listdir(DRIVE_DIR)))

## 2. 패키지 설치
torch는 Colab 기본본 사용(재설치 안 함).

In [ ]:
!pip -q install "transformers==4.48.3" "tokenizers==0.21.4" "accelerate==1.2.1" "bitsandbytes==0.45.5" "peft==0.13.2" json-repair faiss-cpu faster-whisper sentencepiece protobuf fastapi "uvicorn[standard]" python-multipart nest_asyncio

## 3. 코드·인덱스·train.jsonl(Drive) + 어댑터(HF) 배치 (→ /workspace/interview_ai)
`server.py`가 `/workspace/interview_ai/...` 절대경로를 쓰므로 그 구조를 재현. 어댑터는 비공개 HF 레포에서 `HF_TOKEN`(Secrets)으로 받는다.

In [ ]:
import os, shutil
from google.colab import userdata
from huggingface_hub import snapshot_download

HF_TOKEN = userdata.get("HF_TOKEN")   # 🔑 Secrets에 등록한 토큰

APP = "/workspace/interview_ai"
os.makedirs(APP + "/rag", exist_ok=True)

# 1) 코드 + train.jsonl (Drive)
for f in ["server.py", "mapping.py", "voice.py", "stt.py", "train.jsonl"]:
    src = os.path.join(DRIVE_DIR, f)
    assert os.path.exists(src), f"누락: {f} (Drive 최상위에 있는지 확인)"
    shutil.copy(src, os.path.join(APP, f))

# 2) RAG 인덱스 (Drive)
rag_src = os.path.join(DRIVE_DIR, "rag")
for f in os.listdir(rag_src):
    s = os.path.join(rag_src, f)
    if os.path.isfile(s):
        shutil.copy(s, os.path.join(APP, "rag", f))
for must in ["ict_questions.index", "ict_questions.json"]:
    assert os.path.exists(os.path.join(APP, "rag", must)), f"RAG 누락: rag/{must}"

# 3) 어댑터 (비공개 HF 레포 → 토큰으로)
snap = snapshot_download(repo_id="seongchaeae/capstone-interview-ai-lora",
                         allow_patterns=["lora_adapter_v3/*"], token=HF_TOKEN)
shutil.copytree(os.path.join(snap, "lora_adapter_v3"),
                APP + "/lora_adapter_v3", dirs_exist_ok=True)
assert os.path.exists(APP + "/lora_adapter_v3/adapter_model.bin"), "어댑터 .bin 못 받음(HF/토큰 확인)"

print(">>> 배치 완료.")
print("    train.jsonl 줄수:", sum(1 for _ in open(APP + "/train.jsonl", encoding="utf-8")))
print("    rag/:", sorted(os.listdir(APP + "/rag")))
print("    어댑터:", sorted(os.listdir(APP + "/lora_adapter_v3")))

## 4. 서버 기동 (전체 server.py) + 공개 URL
uvicorn으로 `server:app` 실행 → `/health` ready 대기(최대 15분, 첫 다운로드 대비) → cloudflared 공개 URL.

In [ ]:
import subprocess, time, json, urllib.request, threading, re

APP = "/workspace/interview_ai"
log = open("/content/server.log", "w")
srv = subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
                       cwd=APP, stdout=log, stderr=subprocess.STDOUT)
print(">>> 서버 시작. 첫 실행은 모델 다운로드로 7~9분 걸려요. 대기 중...")

def _ready():
    try:
        with urllib.request.urlopen("http://localhost:8000/health", timeout=2) as r:
            return json.loads(r.read()).get("ready") is True
    except Exception:
        return False

ok = False
for i in range(450):  # 최대 ~15분 (첫 다운로드 ~6분 + 로드 여유)
    if srv.poll() is not None:
        print("[실패] 서버가 로딩 중 종료됨. 로그 마지막 2500자:\n")
        print(open("/content/server.log", encoding="utf-8", errors="replace").read()[-2500:])
        break
    if _ready():
        ok = True; break
    if i and i % 30 == 0:
        print(f"    ...로딩 중 ({i*2}s 경과)")
    time.sleep(2)

if ok:
    with urllib.request.urlopen("http://localhost:8000/health", timeout=5) as r:
        print(">>> ✅ 서버 준비 완료:", r.read().decode())
    subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)
    proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    public = None
    for line in proc.stdout:
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
        if m:
            public = m.group(0); break
    threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()
    print("\n========================================")
    print(">>> 공개 URL :", public)
    print(">>> 헬스체크 :", (public or "") + "/health")
    print(">>> Swagger :", (public or "") + "/docs   (전체 엔드포인트 명세)")
    print("========================================")
else:
    print(">>> 준비 실패. 위 로그를 확인하세요.")

## 5. 호출 / 종료

전체 명세는 위 출력의 **`/docs`**(Swagger). 예:
```bash
curl -X POST "<공개URL>/interview/question" -H "Content-Type: application/json" \
  -d '{"topic":"네트워크","k":3,"lang":"ko"}'
```

- 백엔드(Spring)는 이 공개 URL을 Base URL로 호출(계약: repo `deploy/AI_연동가이드.md`).
- **세션제** — Colab 끄면 URL 사라짐. 다시 쓰려면 Run all(첫 다운로드 다시 7~9분).
- STT는 첫 호출 전 whisper 다운로드가 백그라운드로 끝나야 동작.